In [2]:
import xmlrpc.client

# =========================================================================
# 1. ENVIRONMENT CONFIGURATION & AUTHENTICATION
# =========================================================================
URL = "http://localhost:8069"
DB = "m.nushath"       
USERNAME = "user@company.com"  
API_KEY = "user"              

print("Connecting to Odoo XML-RPC API...")
common = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/common', allow_none=True)
models = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/object', allow_none=True)

uid = common.authenticate(DB, USERNAME, API_KEY, {})

if not uid:
    print("[CRITICAL] Authentication failed! Check your credentials.")
    exit()

print(f"[SUCCESS] Connected as User ID: {uid}\n" + "="*60)

Connecting to Odoo XML-RPC API...
[SUCCESS] Connected as User ID: 5


In [3]:
# =========================================================================
# METHOD 1: search()
# Purpose: Find records matching a filter and return ONLY their database IDs.
# =========================================================================
print("\n1. DEMONSTRATING: search()")

# Find up to 3 storable or consumable products sorted by their ID descending
product_ids = models.execute_kw(DB, uid, API_KEY, 'product.template', 'search',
    # Positional Argument [0]: Domain filter array
    [[
        ('type', 'in', ['consu', 'product']),
        ('active', '=', True)
    ]], 
    # Keyword Arguments dictionary
    {
        'limit': 3,
        'order': 'id desc'
    }
)

print(f" -> Execution call parameter structure: search([domain], {{options}})")
print(f" -> Raw returned data type: {type(product_ids)}")
print(f" -> Matching Product Database IDs found: {product_ids}")


1. DEMONSTRATING: search()
 -> Execution call parameter structure: search([domain], {options})
 -> Raw returned data type: <class 'list'>
 -> Matching Product Database IDs found: [11, 10, 5]


In [4]:
# =========================================================================
# METHOD 2: read()
# Purpose: Fetch specific column values for an explicit list of known IDs.
# =========================================================================
print("\n" + "-"*60 + "\n2. DEMONSTRATING: read()")

if not product_ids:
    print(" -> Skipped: No product IDs available from the search method to read.")
else:
    # Read the fields from the IDs extracted in the step above
    product_data = models.execute_kw(DB, uid, API_KEY, 'product.template', 'read',
        # Positional Argument [0]: A strict flat list of integer IDs (Mandatory)
        [product_ids], 
        # Keyword Arguments dictionary
        {
            # Explicitly state fields to save database processing time and memory bandwidth
            'fields': ['name', 'list_price', 'categ_id'] 
        }
    )
    
    print(f" -> Execution call parameter structure: read([id_list], {{fields_dict}})")
    print(f" -> Raw returned data type: {type(product_data)} (List of Dictionaries)")
    
    for item in product_data:
        # Note on relational fields: 'categ_id' (Many2one) returns as a [ID, "Name"] list format natively
        category_name = item['categ_id'][1] if item.get('categ_id') else "Uncategorized"
        print(f"    [*] ID {item['id']}: {item['name']} | Price: ${item['list_price']} | Category: {category_name}")


------------------------------------------------------------
2. DEMONSTRATING: read()
 -> Execution call parameter structure: read([id_list], {fields_dict})
 -> Raw returned data type: <class 'list'> (List of Dictionaries)
    [*] ID 11: Ring | Price: $1.0 | Category: Uncategorized
    [*] ID 10: Watch | Price: $1.0 | Category: Uncategorized
    [*] ID 5: shirt | Price: $30.0 | Category: Uncategorized


In [5]:
# =========================================================================
# METHOD 3: search_read()
# Purpose: Combine search and read into 1 fast network round-trip request.
# =========================================================================
print("\n" + "-"*60 + "\n3. DEMONSTRATING: search_read()")

# Dynamically look up details of up to 2 service-type products directly
service_products = models.execute_kw(DB, uid, API_KEY, 'product.template', 'search_read',
    # Positional Argument [0]: Domain filter array
    [[
        ('type', '=', 'service')
    ]],
    # Keyword Arguments dictionary (Combines both search parameters and read fields)
    {
        'fields': ['name', 'type', 'default_code'],
        'limit': 2
    }
)

print(f" -> Execution call parameter structure: search_read([domain], {{fields_and_limits}})")
print(f" -> Raw returned data type: {type(service_products)}")

for svc in service_products:
    sku = svc.get('default_code') or "NO_SKU"
    print(f"    [*] Product: {svc['name']} | SKU: {sku} | Type: {svc['type']}")


------------------------------------------------------------
3. DEMONSTRATING: search_read()
 -> Execution call parameter structure: search_read([domain], {fields_and_limits})
 -> Raw returned data type: <class 'list'>
    [*] Product: Aramex | SKU: ARAMEX | Type: service
    [*] Product: Mondial Relay | SKU: MR | Type: service


In [6]:
# =========================================================================
# METHOD 4: search_count()
# Purpose: Calculate the numeric count of items matching a filter (Great for metrics).
# =========================================================================
print("\n" + "-"*60 + "\n4. DEMONSTRATING: search_count()")

# Count how many total products are currently archived/inactive in the system
archived_count = models.execute_kw(DB, uid, API_KEY, 'product.template', 'search_count',
    # Positional Argument [0]: Domain filter array
    [[
        ('active', '=', False)
    ]]
    # Keyword Arguments: None accepted by this endpoint method
)

print(f" -> Execution call parameter structure: search_count([domain])")
print(f" -> Raw returned data type: {type(archived_count)}")
print(f" -> Total archived products in system: {archived_count}")


------------------------------------------------------------
4. DEMONSTRATING: search_count()
 -> Execution call parameter structure: search_count([domain])
 -> Raw returned data type: <class 'int'>
 -> Total archived products in system: 0


In [7]:
# =========================================================================
# METHOD 5: fields_get()
# Purpose: Inspect the table schema metadata (labels, choices, field data types).
# =========================================================================
print("\n" + "-"*60 + "\n5. DEMONSTRATING: fields_get()")

# Inspect how the product 'type' field is built internally
type_field_metadata = models.execute_kw(DB, uid, API_KEY, 'product.template', 'fields_get',
    # Positional Argument [0]: Empty array (No domain filter applies here)
    [], 
    # Keyword Arguments dictionary
    {
        'allfields': ['type'],                             # Inspect only the 'type' column
        'attributes': ['string', 'type', 'selection']       # Extract only these property definitions
    }
)

print(f" -> Execution call parameter structure: fields_get([], {{attributes_dict}})")
print(f" -> Raw returned data type: {type(type_field_metadata)}")

if 'type' in type_field_metadata:
    meta = type_field_metadata['type']
    print(f"    [*] Field UI String Label: {meta.get('string')}")
    # selection lists out the dropdown choices: [('technical_key', 'User Label')]
    print(f"    [*] Valid Selection Dropdown Choices Options: {meta.get('selection')}")

print("\n" + "="*60 + "\n[FINISHED] Read orchestration sequence completed successfully.")


------------------------------------------------------------
5. DEMONSTRATING: fields_get()
 -> Execution call parameter structure: fields_get([], {attributes_dict})
 -> Raw returned data type: <class 'dict'>
    [*] Field UI String Label: Product Type
    [*] Valid Selection Dropdown Choices Options: [['consu', 'Goods'], ['service', 'Service'], ['combo', 'Combo']]

[FINISHED] Read orchestration sequence completed successfully.
